# UFC Medallion Architecture

Bronze → Silver → Gold pipeline for the Kaggle UFC Complete Dataset (all events 1996–2024). The bronze layer ingests raw CSV data directly via `kagglehub`, preserving the source schema with ingestion metadata (`_source_file`, `_ingestion_timestamp`).

In [0]:
pip install kagglehub

In [0]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("maksbasher/ufc-complete-dataset-all-events-1996-2024")

print("Path to dataset files:", path)

In [0]:
import kagglehub
import os

# Download latest version (uses cache if already downloaded)
path = kagglehub.dataset_download("maksbasher/ufc-complete-dataset-all-events-1996-2024")
print("Path to dataset files:", path)

print("\nFiles in dataset (recursive):")
for root, dirs, files in os.walk(path):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        rel_path = os.path.relpath(fpath, path)
        print(f"  {rel_path}  ({size_mb:.2f} MB)")

In [0]:
import pandas as pd

# Examine schemas of all CSV files
csv_files = [
    "UFC dataset/Fighter stats/fighter_stats.csv",
    "UFC dataset/Medium set/medium_dataset.csv",
    "UFC dataset/Large set/large_dataset.csv",
    "UFC dataset/Small set/completed_events_small.csv",
    "UFC dataset/Small set/upcoming_events_small.csv",
]

for csv_file in csv_files:
    full_path = os.path.join(path, csv_file)
    table_name = csv_file.split("/")[-1].replace(".csv", "")
    print(f"\n{'='*60}")
    print(f"Table: {table_name}")
    print(f"Path: {csv_file}")
    pdf = pd.read_csv(full_path)
    print(f"Row count: {len(pdf)}")
    print(f"Columns ({len(pdf.columns)}):")
    for col in pdf.columns:
        print(f"  - {col} ({pdf[col].dtype})")
    print("\nSample (2 rows):")
    print(pdf.head(2).to_string())

In [0]:
from pyspark.sql.types import TimestampType
from pyspark.sql import functions as F
from datetime import datetime

# Create bronze schema
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ufc_bronze")
spark.sql("COMMENT ON SCHEMA workspace.ufc_bronze IS 'Bronze layer: raw UFC data loaded from Kaggle dataset (maksbasher/ufc-complete-dataset-all-events-1996-2024)'")

# Define CSV files to load as bronze tables
csv_files = [
    ("UFC dataset/Fighter stats/fighter_stats.csv", "fighter_stats"),
    ("UFC dataset/Medium set/medium_dataset.csv", "medium_dataset"),
    ("UFC dataset/Large set/large_dataset.csv", "large_dataset"),
    ("UFC dataset/Small set/completed_events_small.csv", "completed_events_small"),
    ("UFC dataset/Small set/upcoming_events_small.csv", "upcoming_events_small"),
]

bronze_tables = []

for csv_rel_path, table_name in csv_files:
    full_path = os.path.join(path, csv_rel_path)
    full_table_name = f"workspace.ufc_bronze.{table_name}"
    
    print(f"\nLoading {table_name}...")
    
    # Read CSV with pandas (driver-local file access)
    pdf = pd.read_csv(full_path)
    
    # Convert to Spark DataFrame
    df = spark.createDataFrame(pdf)
    
    # Add bronze metadata columns
    df = df.withColumn("_source_file", F.lit(csv_rel_path)) \
           .withColumn("_ingestion_timestamp", F.lit(datetime.now()).cast(TimestampType()))
    
    # Write as Delta table (overwrite for idempotency)
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)
    
    row_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {full_table_name}").collect()[0]["cnt"]
    print(f"  -> {full_table_name}: {row_count} rows, {len(df.columns)} columns")
    bronze_tables.append((full_table_name, row_count, len(df.columns)))

print("\n" + "="*60)
print("Bronze layer loaded successfully!")
print(f"{'Table':<45} {'Rows':>8} {'Cols':>6}")
print("-" * 60)
for tname, rows, cols in bronze_tables:
    print(f"{tname:<45} {rows:>8,} {cols:>6}")